# EA3 - Actividad 3.2b: Watermarks y Ventanas (Explicacion Visual)

## Objetivos
- Entender **visualmente** que son los watermarks y como funcionan
- Ver el efecto de datos tardios (late data) en ventanas de tiempo
- Simular el comportamiento de Spark Structured Streaming con diferentes configuraciones
- Comparar ventanas fijas (tumbling) vs deslizantes (sliding)

> **NOTA:** Este notebook NO necesita Kafka. Solo usa Python y matplotlib.
> Corre con el perfil **basico** de Docker:
> ```bash
> docker-compose --profile basico up -d
> ```

## Conceptos Clave

### Ventana Temporal (Window)

Una **ventana** agrupa eventos que ocurren en un intervalo de tiempo. Ejemplo: "total de ventas entre las 12:00 y 12:01".

Hay dos tipos principales:

| Tipo | Descripcion | Ejemplo |
|------|-------------|---------|
| **Tumbling Window** | Ventanas fijas, no se superponen | [12:00-12:01], [12:01-12:02], ... |
| **Sliding Window** | Ventanas que se deslizan, se superponen | [12:00-12:05], [12:01-12:06], ... |

```
Tumbling (duracion=1min):
  [12:00 - 12:01] [12:01 - 12:02] [12:02 - 12:03]
       e1,e2            e3                e4

Sliding (duracion=5min, slide=1min):
  [12:00 - 12:05]  e1,e2,e3
  [12:01 - 12:06]  e2,e3,e4
  [12:02 - 12:07]  e3,e4
```

### Watermark

El **watermark** es un mecanismo para manejar datos que llegan **tarde** (late data).

```
Tiempo real:       0s    5s    10s    15s    20s    25s    30s
                   |      |      |      |      |      |      |
Eventos:           e1     e2     e3           e4     e5
                   (t=0)  (t=5)  (t=10)        (t=15) (t=20)

Watermark (10s):         |      |      |      |      |
                   Los eventos con t < watermark
                   ya no se consideran en ventanas
```

### Por que usamos watermarks?

1. **Manejar datos tardios:** No todos los eventos llegan en orden
2. **Liberar memoria:** Spark puede olvidar ventanas viejas
3. **Evitar resultados incorrectos:** Si esperamos para siempre, nunca emitimos resultados

El watermark define: *"?Cuantos segundos esperamos por datos tardios antes de cerrar una ventana?"*

## Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
print("Librerias cargadas.")

## 1. Simular Eventos con Tiempos Variables

Primero, creamos una funcion que genera eventos con timestamps, incluyendo algunos que llegan **tarde** (late data).

In [ ]:
def generar_eventos_con_retraso(n_eventos=30, pct_tardios=0.2, retraso_max_seg=8):
    """
    Genera eventos donde algunos llegan con retraso (late data).

    Parametros:
        n_eventos: Total de eventos
        pct_tardios: Proporcion de eventos que llegan tarde (0.0 a 1.0)
        retraso_max_seg: Maximo retraso en segundos para late data

    Retorna:
        eventos: Lista de dicts con 'id', 'event_time', 'arrival_time', 'monto'
    """
    np.random.seed(42)
    eventos = []
    tiempo_actual = 0

    for i in range(n_eventos):
        # El tiempo del evento avanza ~1 segundo cada vez (con variacion)
        tiempo_actual += max(0.2, np.random.exponential(1.0))
        event_time = tiempo_actual

        # Algunos eventos llegan tarde
        es_tardio = np.random.random() < pct_tardios
        if es_tardio:
            retraso = np.random.uniform(1.0, retraso_max_seg)
            arrival_time = tiempo_actual + retraso
        else:
            arrival_time = tiempo_actual

        eventos.append({
            "id": i + 1,
            "event_time": event_time,
            "arrival_time": arrival_time,
            "monto": np.random.randint(1000, 50000),
            "es_tardio": es_tardio,
            "retraso": arrival_time - event_time,
        })

    return eventos


eventos = generar_eventos_con_retraso(30, pct_tardios=0.25, retraso_max_seg=8)

print(f"Eventos generados: {len(eventos)}")
print(f"Tardios: {sum(1 for e in eventos if e['es_tardio'])}")
print()
print("  ID  | Event Time | Arrival Time | Retraso | Tardio?")
print("-" * 60)
for e in eventos[:15]:
    flag = "*TARDIO*" if e['es_tardio'] else ""
    print(f"  {e['id']:3d} | {e['event_time']:10.2f}s | {e['arrival_time']:10.2f}s | "
          f"{e['retraso']:5.1f}s | {flag}")
print(f"  ... ({len(eventos) - 15} eventos mas)")

## 2. Visualizar Eventos en la Linea de Tiempo

Este grafico muestra **cuando ocurrio** cada evento vs **cuando llego** al sistema.
Los eventos tardios estan en rojo.

In [ ]:
def graficar_linea_tiempo(eventos, tamano_ventana=10, watermark=None):
    """Grafica los eventos en una linea de tiempo."""
    fig, ax = plt.subplots(figsize=(14, 6))

    max_time = max(e['arrival_time'] for e in eventos) + 2

    # Lineas de tiempo
    for e in eventos:
        color = 'red' if e['es_tardio'] else 'steelblue'
        # Linea desde event_time hasta arrival_time
        ax.plot([e['event_time'], e['arrival_time']], [e['id'], e['id']],
                color=color, alpha=0.3, linewidth=1)
        # Circulo en event_time
        ax.scatter(e['event_time'], e['id'], color=color, s=60, zorder=5,
                  edgecolors='white', linewidth=0.5)
        # Triangulo en arrival_time
        ax.scatter(e['arrival_time'], e['id'], marker='v', color=color,
                  s=80, zorder=5, alpha=0.7)

    # Ventanas fijas (tumbling)
    ventanas = np.arange(0, max_time + tamano_ventana, tamano_ventana)
    for i, v in enumerate(ventanas[:-1]):
        ax.axvspan(v, v + tamano_ventana, alpha=0.05, color=f'C{i % 10}')
        ax.text(v + tamano_ventana / 2, -1.5, f'V{i + 1}\n[{v:.0f}-{v + tamano_ventana:.0f}s]',
                ha='center', fontsize=9, color=f'C{i % 10}')

    # Watermark
    if watermark is not None:
        tiempo_actual = max(e['arrival_time'] for e in eventos)
        watermark_time = tiempo_actual - watermark
        ax.axvline(watermark_time, color='red', linewidth=2, linestyle='--',
                  label=f'Watermark ({watermark}s de retraso)')
        ax.axvspan(0, watermark_time, alpha=0.1, color='red',
                  label='Datos descartados por watermark')

    ax.set_xlabel('Tiempo (segundos)', fontsize=12)
    ax.set_ylabel('Evento ID', fontsize=12)
    ax.set_title('Linea de Tiempo de Eventos: Circulo = ocurrio, Triangulo = llego', fontsize=14)
    ax.set_ylim(-2, len(eventos) + 2)

    # Leyenda
    leyenda = [
        mpatches.Patch(color='steelblue', label='A tiempo'),
        mpatches.Patch(color='red', label='Tardio (late data)'),
    ]
    if watermark is not None:
        leyenda.append(plt.Line2D([0], [0], color='red', linestyle='--',
                                 linewidth=2, label=f'Watermark ({watermark}s)'))
    ax.legend(handles=leyenda, loc='upper right')

    plt.tight_layout()
    plt.show()


print("Eventos sin watermark (todos se consideran):")
graficar_linea_tiempo(eventos, tamano_ventana=10)

### Interpretacion:

- Los **circulos azules** muestran cuando ocurrio el evento
- Los **triangulos rojos** muestran cuando llego al sistema
- Las **lineas conectan** un mismo evento: si es larga, llego muy tarde
- Las **franjas de colores** son ventanas de 10 segundos
- Los eventos en **rojo** son tardios (late data)

## 3. ?Como Afecta el Watermark a las Ventanas?

Ahora veamos el efecto del watermark. Cuando el watermark avanza, los eventos que lleguen **despues** del watermark seran descartados de las ventanas correspondientes.

In [ ]:
def simular_streaming_con_watermark(eventos, duracion_ventana=10, watermark_seg=5, avanzar_cada=2):
    """
    Simula la llegada de eventos en el tiempo, mostrando como el watermark
    descarta datos tardios de ventanas cerradas.

    Parametros:
        eventos: Lista de eventos con event_time, arrival_time, monto
        duracion_ventana: Segundos de cada ventana
        watermark_seg: Cuantos segundos despues del tiempo actual se descartan datos
        avanzar_cada: Cada cuantos segundos simulados avanza la simulacion
    """
    eventos = sorted(eventos, key=lambda e: e['arrival_time'])
    max_time = max(e['arrival_time'] for e in eventos)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- Estado de la simulacion ---
    ventanas = defaultdict(lambda: {"montos": [], "aceptados": [], "tardios": []})
    eventos_procesados = []
    idx_evento = 0

    for tiempo_actual in np.arange(0, max_time + avanzar_cada, avanzar_cada):
        # Llegan nuevos eventos cuyo arrival_time <= tiempo_actual
        while idx_evento < len(eventos) and eventos[idx_evento]['arrival_time'] <= tiempo_actual:
            e = eventos[idx_evento]

            # A que ventana pertenece este evento?
            inicio_ventana = (e['event_time'] // duracion_ventana) * duracion_ventana
            fin_ventana = inicio_ventana + duracion_ventana
            key_ventana = f"V{int(inicio_ventana)}-{int(fin_ventana)}"

            # Verificar si el evento llego antes del watermark
            # El watermark = tiempo_actual - watermark_seg
            watermark_time = tiempo_actual - watermark_seg

            if e['event_time'] >= watermark_time:
                ventanas[key_ventana]["aceptados"].append(e)
            else:
                ventanas[key_ventana]["tardios"].append(e)

            ventanas[key_ventana]["montos"].append(e['monto'])
            eventos_procesados.append(e)
            idx_evento += 1

    # --- Grafico 1: Linea de tiempo con watermark actual ---
    ax = axes[0]
    for e in eventos:
        color = 'red' if e['es_tardio'] else 'steelblue'
        ax.plot([e['event_time'], e['arrival_time']], [e['id'], e['id']],
                color=color, alpha=0.2, linewidth=1)
        ax.scatter(e['event_time'], e['id'], color=color, s=40,
                  edgecolors='white', linewidth=0.5)

    # Ventanas
    for i, v in enumerate(np.arange(0, max_time + duracion_ventana, duracion_ventana)):
        ax.axvspan(v, v + duracion_ventana, alpha=0.05, color=f'C{i % 10}')

    # Watermark final
    watermark_final = max_time - watermark_seg
    ax.axvline(watermark_final, color='red', linewidth=2, linestyle='--')
    ax.axvspan(0, watermark_final, alpha=0.1, color='red')
    ax.set_title(f'Eventos con Watermark de {watermark_seg}s')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Evento ID')

    # --- Grafico 2: Monto por ventana (aceptados) ---
    ax = axes[1]
    ventanas_ordenadas = sorted(ventanas.keys())
    x_pos = np.arange(len(ventanas_ordenadas))
    aceptados = [len(ventanas[v]["aceptados"]) for v in ventanas_ordenadas]
    tardios = [len(ventanas[v]["tardios"]) for v in ventanas_ordenadas]

    ax.bar(x_pos, aceptados, label='Aceptados', color='steelblue')
    ax.bar(x_pos, tardios, bottom=aceptados, label='Descartados (tardios)', color='red', alpha=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(ventanas_ordenadas, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Eventos por Ventana (Watermark={watermark_seg}s)')
    ax.set_ylabel('Numero de Eventos')
    ax.legend()

    # --- Grafico 3: Total acumulado vs ventana ---
    ax = axes[2]
    montos_ventana = [sum(v["montos"]) for v in ventanas.values()]
    ax.bar(x_pos, montos_ventana, color='green', alpha=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(ventanas_ordenadas, rotation=45, ha='right', fontsize=8)
    ax.set_title('Monto Total por Ventana')
    ax.set_ylabel('Monto ($)')

    plt.tight_layout()
    plt.show()

    # --- Tabla resumen ---
    print(f"\n{'='*70}")
    print(f"  RESUMEN: Watermark de {watermark_seg}s | Ventanas de {duracion_ventana}s")
    print(f"{'='*70}")
    print(f"  {'Ventana':20s} {'Aceptados':10s} {'Tardios':10s} {'Monto Total':15s}")
    print(f"  {'-'*55}")
    for v in ventanas_ordenadas:
        info = ventanas[v]
        print(f"  {v:20s} {len(info['aceptados']):10d} {len(info['tardios']):10d} "
              f"${sum(info['montos']):>10,}")
    print(f"  {'-'*55}")
    total_a = sum(len(ventanas[v]["aceptados"]) for v in ventanas_ordenadas)
    total_t = sum(len(ventanas[v]["tardios"]) for v in ventanas_ordenadas)
    print(f"  {'TOTAL':20s} {total_a:10d} {total_t:10d}")
    print(f"\n  Eventos descartados por el watermark: {total_t} de {total_a + total_t}")
    if total_a + total_t > 0:
        print(f"  Tasa de descarte: {total_t / (total_a + total_t) * 100:.1f}%")

    return ventanas


print("Simulacion con watermark de 5 segundos:")
ventanas = simular_streaming_con_watermark(eventos, duracion_ventana=10, watermark_seg=5)

### Interpretacion:

1. **Grafico izquierdo:** Muestra la linea de tiempo. La zona roja son datos que el watermark ya descarto
2. **Grafico central:** Cada ventana muestra cuantos eventos fueron aceptados (azul) y cuantos descartados por llegar tarde (rojo)
3. **Grafico derecho:** Monto economico que se pierde si descartamos datos tardios

> **Pregunta:** ?Que pasaria si aumentamos el watermark a 10 segundos? ?Y si lo bajamos a 2 segundos?

## 4. Comparar Diferentes Watermarks

Veamos el efecto de cambiar el watermark: uno muy pequeno descarta muchos datos, uno muy grande retiene datos por mucho tiempo.

In [ ]:
def comparar_watermarks(eventos, duracion_ventana=10, watermarks=[2, 5, 10, 20]):
    """Compara el efecto de diferentes watermarks."""
    resultados = []

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for idx, watermark_seg in enumerate(watermarks):
        ax = axes[idx]

        # Simular con este watermark
        ventanas = defaultdict(lambda: {"aceptados": [], "tardios": []})
        eventos_ordenados = sorted(eventos, key=lambda e: e['arrival_time'])
        max_time = max(e['arrival_time'] for e in eventos)

        for e in eventos_ordenados:
            inicio_v = (e['event_time'] // duracion_ventana) * duracion_ventana
            fin_v = inicio_v + duracion_ventana
            key = f"V{int(inicio_v)}-{int(fin_v)}"
            watermark_actual = e['arrival_time'] - watermark_seg

            if e['event_time'] >= watermark_actual:
                ventanas[key]["aceptados"].append(e)
            else:
                ventanas[key]["tardios"].append(e)

        # Grafico de barras apiladas
        ventanas_ord = sorted(ventanas.keys())
        x = np.arange(len(ventanas_ord))
        a = [len(ventanas[v]["aceptados"]) for v in ventanas_ord]
        t = [len(ventanas[v]["tardios"]) for v in ventanas_ord]

        ax.bar(x, a, label='Aceptados', color='steelblue')
        ax.bar(x, t, bottom=a, label='Tardios', color='red', alpha=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels(ventanas_ord, rotation=45, ha='right', fontsize=7)
        ax.set_title(f'Watermark = {watermark_seg}s | Total: {sum(a)} acep, {sum(t)} tard')
        ax.set_ylabel('Eventos')
        if idx == 0:
            ax.legend(loc='upper right')

        total_a = sum(a)
        total_t = sum(t)
        resultados.append((watermark_seg, total_a, total_t,
                          total_t / (total_a + total_t) * 100 if (total_a + total_t) > 0 else 0))

    plt.tight_layout()
    plt.show()

    # Tabla comparativa
    print(f"\n{'='*60}")
    print(f"  COMPARACION DE WATERMARKS")
    print(f"{'='*60}")
    print(f"  {'Watermark':12s} {'Aceptados':12s} {'Tardios':12s} {'% Descarte':12s}")
    print(f"  {'-'*48}")
    for w, a, t, p in resultados:
        print(f"  {w:5d}s      {a:8d}      {t:8d}      {p:6.1f}%")
    print(f"  {'-'*48}")


comparar_watermarks(eventos, duracion_ventana=10)

### Conclusion Clave:

- **Watermark pequeno (2s):** Descarta muchos datos, resultados rapidos pero posiblemente incompletos
- **Watermark grande (20s):** Retiene casi todos los datos, pero gasta mas memoria y el resultado tarda mas
- **No hay watermark perfecto:** Depende de la tolerancia al retraso de tu aplicacion
  - Fraude bancario: watermark pequeno (reaccion rapida)
  - Reportes financieros: watermark grande (precision ante todo)

## 5. Tumbling vs Sliding Windows

Veamos la diferencia visual entre ventanas fijas (tumbling) y deslizantes (sliding).

In [ ]:
def visualizar_tipos_ventana():
    """Visualiza la diferencia entre tumbling y sliding windows."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    # Eventos de ejemplo
    tiempos_eventos = [2, 4, 7, 8, 11, 13, 16, 18, 22, 25]
    montos = [10000, 25000, 15000, 30000, 12000, 18000, 22000, 35000, 16000, 29000]

    for row, (titulo, duracion, slide) in enumerate([
        ("Tumbling Window (duracion=10s, sin superposicion)", 10, 10),
        ("Sliding Window (duracion=10s, slide=5s, 50% superposicion)", 10, 5),
    ]):
        ax = axes[row]

        # Dibujar eventos
        for i, (t, m) in enumerate(zip(tiempos_eventos, montos)):
            ax.scatter(t, 0, s=m / 500, color='steelblue', zorder=5,
                      edgecolors='white', linewidth=1)
            ax.text(t, 0.3, f'${m:,}', ha='center', fontsize=8, rotation=45)

        # Dibujar ventanas
        ventanas_inicio = np.arange(0, 30, slide)
        colores = plt.cm.Set2(np.linspace(0, 1, len(ventanas_inicio)))

        for i, inicio in enumerate(ventanas_inicio):
            fin = inicio + duracion
            if fin > 30:
                continue

            # Calcular monto en esta ventana
            monto_ventana = sum(
                m for t, m in zip(tiempos_eventos, montos)
                if inicio <= t < fin
            )

            y_pos = -1.5 - i * 0.6
            ax.barh(y_pos, fin - inicio, left=inicio, height=0.4,
                   color=colores[i], alpha=0.7, edgecolor='gray')
            ax.text((inicio + fin) / 2, y_pos, f'${monto_ventana:,}',
                   ha='center', va='center', fontsize=8, fontweight='bold')

        ax.set_title(titulo, fontsize=12, fontweight='bold')
        ax.set_xlim(0, 30)
        ax.set_ylim(-len(ventanas_inicio) - 2, 2)
        ax.set_xlabel('Tiempo (s)')
        ax.set_yticks([])
        ax.axhline(y=0, color='gray', linewidth=1)

        # Eventos como texto
        ax.text(-1.5, 0, 'Eventos:', ha='right', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()


    print("\nDiferencia clave:")
    print("  - Tumbling: Cada evento pertenece a UNA sola ventana")
    print("  - Sliding:  Cada evento puede pertenecer a VARIAS ventanas")
    print("  - Sliding da resultados mas suaves pero requiere mas computo")


visualizar_tipos_ventana()

---
## Ejercicios

In [ ]:
# =============================================================
# EJERCICIO 1: Simular con diferentes watermarks
# =============================================================
# TODO: Usa la funcion generar_eventos_con_retraso() para crear
#   un conjunto de 50 eventos con 30% de tardios y retraso maximo
#   de 15 segundos.
#
# Luego ejecuta simular_streaming_con_watermark() con:
#   - watermark = 3s
#   - watermark = 15s
#
# ?Que observas? ?Cuantos eventos se descartan en cada caso?
#
# Pistas:
#   - generar_eventos_con_retraso(50, pct_tardios=0.3, retraso_max_seg=15)
#   - simular_streaming_con_watermark(eventos, watermark_seg=3)
#   - simular_streaming_con_watermark(eventos, watermark_seg=15)

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Analizar el impacto economico del watermark
# =============================================================
# TODO: Usa los eventos generados en el ejercicio anterior.
#
# Para cada watermark en [1, 3, 5, 10, 15, 30]:
#   1. Simula el streaming
#   2. Calcula el monto total ACEPTADO (no descartado)
#   3. Calcula el porcentaje del monto total que se pierde
#
# Grafica:
#   - Eje X: watermark (segundos)
#   - Eje Y: % del monto total aceptado
#   - Titulo: "Impacto del Watermark en la Captura de Ingresos"
#
# Pregunta: Si cada 1% de ventas perdidas equivale a $10,000 USD,
#   ?cuanto "pierde" la empresa con cada configuracion de watermark?
#
# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Ventana deslizante manual
# =============================================================
# TODO: Implementa una ventana deslizante (sliding window)
#   manualmente con las siguientes caracteristicas:
#
#   - Duracion de ventana: 6 segundos
#   - Slide: 2 segundos (se actualiza cada 2s)
#   - Cada ventana cubre 6s: [0-6], [2-8], [4-10], [6-12], ...
#
# Datos de prueba:
tiempos = [1, 3, 4, 7, 9, 10, 12, 14, 15, 18, 20, 22]
montos  = [5000, 12000, 8000, 15000, 6000, 20000, 9000, 14000, 11000, 18000, 7000, 16000]
#
# Para cada ventana, calcula:
#   - Cuantos eventos caen dentro
#   - Monto total
#   - Monto promedio
#
# Formato de salida:
#   Ventana [0-6]:   3 eventos, $25,000 total, $8,333 prom
#   Ventana [2-8]:   4 eventos, $41,000 total, $10,250 prom
#   ...
#
# Pista: Para cada ventana:
#   inicio = i * slide
#   fin = inicio + duracion
#   eventos_en_ventana = [(t,m) for t,m in zip(tiempos,montos) if inicio <= t < fin]

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Que es un watermark:** Limite de tiempo para aceptar datos tardios
2. **Como afecta a las ventanas:** Datos que llegan despues del watermark se descartan
3. **Tumbling vs Sliding:** Ventanas sin superposicion vs con superposicion
4. **Trade-off del watermark:** Muy pequeno pierde datos, muy grande gasta memoria
5. **Aplicacion practica:** Elegir el watermark segun el caso de uso (fraude vs reportes)

---
## Desafio Extra (Opcional)

**Simulador interactivo de watermark:**

Crea una funcion que permita al usuario cambiar el watermark y ver el efecto en tiempo real.
Usa `input()` para preguntar: "?Que watermark quieres probar (en segundos)?"
y muestra los 3 graficos (linea de tiempo, barras apiladas, monto por ventana).

In [ ]:
# =============================================================
# DESAFIO: Simulador interactivo de watermark
# =============================================================
# TODO: Crea un bucle que:
#   1. Pregunte "Watermark (s) [Enter=salir]: "
#   2. Genere nuevos eventos con generar_eventos_con_retraso()
#   3. Ejecute simular_streaming_con_watermark()
#   4. Repita hasta que el usuario presione Enter sin valor
#
# Pista:
#   while True:
#       entrada = input("Watermark (s) [Enter=salir]: ")
#       if not entrada:
#           break
#       wm = int(entrada)
#       eventos = generar_eventos_con_retraso(25, ...)
#       simular_streaming_con_watermark(eventos, watermark_seg=wm)

# Escribe tu codigo aqui:
